In [1]:
# from __future__ import annotations
import numpy as np
import pandas as pd


def generate_synthetic_ehr(
    num_samples: int,
    positive_rate: float,
    seed: int
) -> pd.DataFrame:

    rng = np.random.default_rng(seed)

    # ------------------- Basic Demographics -------------------
    age = rng.integers(40, 90, size=num_samples)
    sex = rng.choice(["M", "F"], size=num_samples)

    # ------------------- Metabolic & Pancreatic Features -------------------
    bmi = rng.normal(27.0, 4.5, size=num_samples).clip(16, 55)
    hba1c = rng.normal(5.8, 0.7, size=num_samples).clip(4.5, 12)
    ldl = rng.normal(120, 35, size=num_samples).clip(40, 260)
    ast = rng.normal(25, 10, size=num_samples).clip(5, 150)
    alt = rng.normal(25, 12, size=num_samples).clip(5, 200)

    # ------------------- Neurodegenerative Features -------------------
    apoe_e4 = rng.choice([0, 1], size=num_samples, p=[0.75, 0.25])
    memory_score = rng.normal(7.0, 2.0, size=num_samples).clip(1, 10)
    sleep_hours = rng.normal(6.5, 1.5, size=num_samples).clip(3, 10)
    vitamin_b12 = rng.normal(400, 80, size=num_samples).clip(150, 800)
    depression_score = rng.normal(3.0, 2.0, size=num_samples).clip(0, 10)
    family_history_neuro = rng.choice([0, 1], size=num_samples, p=[0.8, 0.2])

    # ------------------- Risk Signal Calculation -------------------
    risk_logit = (
        0.03 * (age - 60)
        + 0.04 * (hba1c - 6.0)
        + 0.02 * (bmi - 28.0)
        + 0.01 * (ldl - 130)
        + 0.015 * (ast - 25)
        + 0.015 * (alt - 25)
        + 0.4 * apoe_e4
        + 0.03 * (7 - memory_score)
        + 0.02 * (7 - sleep_hours)
        + 0.015 * depression_score
        + 0.02 * family_history_neuro
        + rng.normal(0, 0.4, size=num_samples)
    )

    # ------------------- Label Calibration -------------------
    threshold = np.quantile(risk_logit, 1 - positive_rate)
    label = (risk_logit >= threshold).astype(int)

    # ------------------- Final DataFrame -------------------
    df = pd.DataFrame({
        "age": age,
        "sex": sex,
        "bmi": bmi,
        "hba1c": hba1c,
        "ldl": ldl,
        "ast": ast,
        "alt": alt,
        "apoe_e4": apoe_e4,
        "memory_score": memory_score,
        "sleep_hours": sleep_hours,
        "vitamin_b12": vitamin_b12,
        "depression_score": depression_score,
        "family_history_neuro": family_history_neuro,
        "label": label
    })

    return df


# ============================================================
# FUNCTION CALL + DATA SAVE
# ============================================================

if __name__ == "__main__":

    # Generate dataset
    df = generate_synthetic_ehr(
        num_samples=5000,
        positive_rate=0.3,   # 30% positive class
        seed=42
    )

    # Save to CSV
    df.to_csv("synthetic_medical_ehr.csv", index=False)